# Taller 09. OCR de manuscritos: modelos de visión y lenguaje contra OCR clásicos

**Qué hace:** transcribe cinco manuscritos españoles antiguos (de la Edad Media al siglo XVIII) con cinco sistemas, y mide su error contra una transcripción humana:

- dos OCR clásicos, entrenados con letra impresa: **docTR** y **Tesseract**;
- tres modelos de visión y lenguaje de pesos abiertos: **Churro-3B**, entrenado con documentos históricos, incluidos manuscritos; y **LightOnOCR-3-4B** y **Chandra OCR 2**, especializados en OCR.

**Para qué:** ver la distancia entre los dos enfoques en el caso más difícil, y discutir qué significa «error» cuando la referencia es una transcripción paleográfica.

**Por qué estos modelos:** son los tres mejores modelos abiertos de nuestra evaluación de manuscritos (30 páginas, 12 sistemas) que entran en una GPU T4. Ver `informe-ocr-berrutti.md`, sección 8.

**Datos:** cinco páginas del split de prueba de [CHURRO-DS](https://huggingface.co/datasets/stanford-oval/churro-dataset) (Stanford), las más legibles de nuestro set de manuscritos. El cuaderno descarga un archivo de unos 880 MB. Licencia: **solo uso de investigación**. Los pesos de Churro-3B tienen la licencia de investigación de Qwen.

**Entorno:** Colab con GPU T4 (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*), y después *Entorno de ejecución → Ejecutar todas*. Sin GPU funciona, pero el modelo de visión y lenguaje tarda varios minutos por página.

## 0. Instalar dependencias (solo Colab)

La primera ejecución descarga además los pesos de los tres modelos (unos 25 GB en total): calcular entre 10 y 15 minutos.

In [ ]:
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run("apt-get -qq install -y tesseract-ocr > /dev/null", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.5.4", "accelerate", "bitsandbytes",
                    "python-doctr", "pytesseract", "jiwer"], check=True)
    print("Instalados Tesseract, transformers, docTR y jiwer")
else:
    print("Fuera de Colab no se instala nada: ver los paquetes en cuadernos/README.md.")

## 1. Funciones de texto

`limpiar()` prepara cualquier transcripción (la referencia y las de cada sistema) para compararlas con las mismas reglas.

`normalizar()` aplica además una normalización gráfica, habitual al evaluar manuscritos antiguos transcriptos de forma diplomática.

In [ ]:
import html
import re


def limpiar(texto):
    """Deja solo el texto: sin marcas XML, HTML o Markdown, con las palabras cortadas
    por guion al final de línea unidas, y con un solo espacio entre palabras."""
    texto = re.sub(r"<Metadata>.*?</Metadata>", "", texto, flags=re.S)  # ficha que Churro agrega antes del texto
    texto = re.sub(r"</(Line|p|div|li|h\d)>|<br\s*/?>|<Gap[^>]*/>", " ", texto)  # fin de línea o de párrafo
    texto = re.sub(r"<[^>]+>", "", texto)
    texto = html.unescape(texto).replace("**", "")
    texto = re.sub(r"^#+\s*", "", texto, flags=re.M)
    texto = re.sub(r"(\w)-\s+(?=[a-záéíóúüñ])", r"\1", texto)  # "pala- bra" -> "palabra"
    return re.sub(r"\s+", " ", texto).strip()

import unicodedata

LETRAS = str.maketrans({"ſ": "s", "ꝛ": "r", "ꝺ": "d", "ꝑ": "p", "ꝓ": "p", "ꝗ": "q", "ꝙ": "q", "⁊": "e",
                        "&": "e", "v": "u", "j": "i", "y": "i", "ß": "ss", "æ": "ae", "œ": "oe"})


def normalizar(texto):
    """Normalización gráfica para escrituras antiguas: minúsculas, sin tildes ni signos de
    abreviatura, sin puntuación, letras antiguas llevadas a las modernas y u/v, i/j/y unificadas.
    Las abreviaturas NO se desarrollan: "q̃" sigue siendo distinto de "que"."""
    texto = re.sub(r"(\w)[-=¬]\s+(?=\w)", r"\1", texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^\w\s]|_", " ", texto.translate(LETRAS))
    return re.sub(r"\s+", " ", texto).strip()

## 2. Cargar las páginas

Se descarga del dataset el archivo que contiene las cinco páginas y se leen solo esas filas. La referencia es la transcripción del dataset, **diplomática**: conserva grafías, abreviaturas y signos tal como están en el original. Los tramos ilegibles que el editor marcó como faltantes no se cuentan.

In [ ]:
import io

import pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image

ARCHIVO = "data/test-00000-of-00003.parquet"
PAGINAS = [
    "codea/CODEA-3577",  # documento notarial (corpus CODEA)
    "spain/bnf-esp-256_fuero_juzgo_6v",  # Fuero Juzgo, BnF Espagnol 256, f. 6v (códice medieval)
    "notarial/170025120000003,0052",  # protocolo notarial
    "codea/CODEA-2561",  # documento notarial (corpus CODEA)
    "aldicam/ALDICAM-0341",  # documento de archivo (corpus ALDICAM)
]

parquet = hf_hub_download("stanford-oval/churro-dataset", ARCHIVO, repo_type="dataset")
filas = pd.read_parquet(parquet, columns=["example_id", "image", "cleaned_transcription"],
                        filters=[("example_id", "in", PAGINAS)])

paginas = []
for _, fila in filas.iterrows():
    imagen = Image.open(io.BytesIO(fila["image"]["bytes"])).convert("RGB")
    imagen.thumbnail((2048, 2048))  # el lado mayor queda en 2048 píxeles, como en la evaluación
    # La referencia es el texto de cada <Line> de la transcripción XML.
    lineas = re.findall(r"<Line\b[^>]*>(.*?)</Line>", fila["cleaned_transcription"], re.S)
    paginas.append({"id": fila["example_id"], "imagen": imagen, "referencia": limpiar(" ".join(lineas))})
paginas.sort(key=lambda p: PAGINAS.index(p["id"]))
print(len(paginas), "páginas")

In [ ]:
from IPython.display import display

for pagina in paginas:
    miniatura = pagina["imagen"].copy()
    miniatura.thumbnail((600, 600))
    print(pagina["id"], "-", len(pagina["referencia"]), "caracteres en la referencia")
    display(miniatura)

## 3. OCR clásicos: docTR y Tesseract

Son la **línea de base**. Los dos trabajan en dos pasos: primero detectan dónde hay texto y después reconocen cada palabra o línea. No usan el contexto de la página para «adivinar» lo que no se ve bien.

- [docTR](https://github.com/mindee/doctr) (Mindee): redes neuronales de detección y reconocimiento, entrenadas con documentos modernos.
- [Tesseract](https://github.com/tesseract-ocr/tesseract) (Google, hoy proyecto abierto): el OCR libre más usado. Se usa con el modelo de español más preciso (`tessdata_best`) y tratando la imagen como un solo bloque de texto (`--psm 6`), la configuración que mejor funcionó en nuestra evaluación.

In [ ]:
import time
import urllib.request
from pathlib import Path

import numpy as np
import pytesseract
import torch
from doctr.models import ocr_predictor

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

# docTR: detección db_resnet50 + reconocimiento crnn_vgg16_bn
modelo_doctr = ocr_predictor(pretrained=True).to(DISPOSITIVO)


def transcribir_doctr(imagen):
    pagina = modelo_doctr([np.array(imagen)]).export()["pages"][0]
    lineas = [" ".join(palabra["value"] for palabra in linea["words"])
              for bloque in pagina["blocks"] for linea in bloque["lines"]]
    return "\n".join(lineas)


# Tesseract con el modelo de español más preciso (tessdata_best) y --psm 6: un bloque de texto uniforme.
TESSDATA = Path("tessdata")
if not (TESSDATA / "spa.traineddata").exists():
    TESSDATA.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://github.com/tesseract-ocr/tessdata_best/raw/main/spa.traineddata",
                               TESSDATA / "spa.traineddata")


def transcribir_tesseract(imagen):
    return pytesseract.image_to_string(imagen, lang="spa", config=f"--tessdata-dir {TESSDATA} --psm 6")


print("Tesseract", pytesseract.get_tesseract_version(), "- docTR en", DISPOSITIVO)

## 4. Modelos de visión y lenguaje

| Modelo | Tipo | Parámetros | CER normalizado mediano en la evaluación |
|---|---|---|---|
| [Churro-3B](https://huggingface.co/stanford-oval/churro-3B) (Stanford) | Qwen2.5-VL-3B ajustado con 100.000 páginas históricas, impresas y manuscritas; devuelve XML | 3.800 millones | 8,9 % |
| [LightOnOCR-3-4B](https://huggingface.co/lightonai/LightOnOCR-3-4B) (LightOn) | especializado en OCR; devuelve Markdown | 4.000 millones | 13,8 % |
| [Chandra OCR 2](https://huggingface.co/datalab-to/chandra-ocr-2) (Datalab) | especializado en OCR; devuelve HTML | 5.300 millones | 19,8 % |

Cada modelo usa la instrucción y el muestreo de su fabricante.

Quedaron afuera dots.mocr (17,6 %), porque su código exige FlashAttention, que no funciona en la T4; olmOCR 2 (17,8 %), que solo entra en la T4 cuantizado a 4 bits y así pierde unos diez puntos; y los modelos de propósito general, como Qwen3-VL-8B, que rindieron mal en manuscritos (19,4 %) y a veces entraron en bucles de repetición.

In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

# bfloat16 en GPU modernas (L4, A100); float16 en la T4, que no tiene bfloat16 nativo; float32 sin GPU.
if not torch.cuda.is_available():
    TIPO = torch.float32
elif torch.cuda.get_device_capability()[0] >= 8:
    TIPO = torch.bfloat16
else:
    TIPO = torch.float16


def cargar(repo, cuantizar=False, codigo_propio=False, **_):
    opciones = {"dtype": TIPO, "device_map": DISPOSITIVO, "trust_remote_code": codigo_propio}
    if cuantizar:  # 4 bits (NF4): el modelo ocupa la cuarta parte de memoria
        opciones["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                             bnb_4bit_compute_dtype=TIPO)
    clase = AutoModelForCausalLM if codigo_propio else AutoModelForImageTextToText
    return clase.from_pretrained(repo, **opciones), AutoProcessor.from_pretrained(repo, trust_remote_code=codigo_propio)


def transcribir(modelo, procesador, imagen, sistema=None, instruccion=None, muestreo=None, plantilla=None, **_):
    contenido = [{"type": "image", "image": imagen}]
    if instruccion:
        contenido.append({"type": "text", "text": instruccion})
    mensajes = [{"role": "user", "content": contenido}]
    if sistema:
        mensajes.insert(0, {"role": "system", "content": [{"type": "text", "text": sistema}]})
    entradas = procesador.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=True, return_dict=True,
                                              return_tensors="pt", **(plantilla or {})).to(modelo.device)
    # Todos estos modelos derivan de Qwen y terminan cada respuesta con <|im_end|>.
    fin = procesador.tokenizer.convert_tokens_to_ids("<|im_end|>")
    salida = modelo.generate(**entradas, max_new_tokens=MAX_TOKENS, eos_token_id=fin, **(muestreo or {"do_sample": False}))
    return procesador.decode(salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True)

**Cómo se usan.** `cargar()` descarga un modelo y lo pone en la GPU, cuantizado a 4 bits si se pide. `transcribir()` arma el mensaje (imagen, instrucción de sistema e instrucción de usuario, según el modelo), genera el texto y lo devuelve crudo.

In [ ]:
MAX_TOKENS = 4000  # tope de texto generado por página; corta los bucles de repetición

# Final de la instrucción oficial de Chandra (repositorio datalab-to/chandra).
CHANDRA_FIN = """Only use these tags ['math', 'br', 'i', 'b', 'u', 'del', 'sup', 'sub', 'table', 'tr', 'td', 'p', 'th', 'div', 'pre', 'h1', 'h2', 'h3', 'h4', 'h5', 'ul', 'ol', 'li', 'input', 'a', 'span', 'img', 'hr', 'tbody', 'small', 'caption', 'strong', 'thead', 'big', 'code', 'chem'], and these attributes ['class', 'colspan', 'rowspan', 'display', 'checked', 'type', 'border', 'value', 'style', 'href', 'alt', 'align', 'data-bbox', 'data-label'].

Guidelines:
* Inline math: Surround math with <math>...</math> tags. Math expressions should be rendered in KaTeX-compatible LaTeX. Use display for block math.
* Tables: Use colspan and rowspan attributes to match table structure.
* Formatting: Maintain consistent formatting with the image, including spacing, indentation, subscripts/superscripts, and special characters.
* Images: Include a description of any images in the alt attribute of an <img> tag. Do not fill out the src property. Describe in detail inside the div tag. Also convert charts to high fidelity data, and convert diagrams to mermaid.
* Forms: Mark checkboxes and radio buttons properly.
* Text: join lines together properly into paragraphs using <p>...</p> tags.  Use <br> tags for line breaks within paragraphs, but only when absolutely necessary to maintain meaning.
* Chemistry: Use <chem>...</chem> tags for chemical formulas with reactive SMILES.
* Lists: Preserve indents and proper list markers.
* Use the simplest possible HTML structure that accurately represents the content of the block.
* Make sure the text is accurate and easy for a human to read and interpret.  Reading order should be correct and natural."""

# Borrá una entrada para no correr ese modelo. Se cargan de a uno: en una T4 no entran dos juntos.
MODELOS = {
    "Churro-3B": {"repo": "stanford-oval/churro-3B",
                  "sistema": "Transcribe the entirety of this historical document to XML format."},
    "LightOnOCR-3-4B": {"repo": "lightonai/LightOnOCR-3-4B",  # sin instrucción: solo transcribe
                        "plantilla": {"enable_thinking": False},
                        "muestreo": {"do_sample": True, "temperature": 0.2, "top_p": 0.9}},
    "Chandra OCR 2": {"repo": "datalab-to/chandra-ocr-2", "instruccion": "OCR this image to HTML.\n\n" + CHANDRA_FIN,
                      "plantilla": {"enable_thinking": False}},
}

## 5. Transcribir

Cada página pasa por los cinco sistemas y se mide cuántos segundos tarda cada uno. Los modelos de visión y lenguaje se cargan de a uno y se liberan antes del siguiente. En una T4, los OCR clásicos tardan alrededor de un segundo por página; los modelos de visión y lenguaje, entre medio minuto y un par de minutos.

In [ ]:
filas = []


def medir(sistema, transcribir_pagina):
    for pagina in paginas:
        inicio = time.perf_counter()
        texto = transcribir_pagina(pagina["imagen"])
        segundos = time.perf_counter() - inicio
        filas.append({"pagina": pagina["id"], "sistema": sistema, "segundos": round(segundos, 1), "texto": texto})
        print(f"{sistema:>22}  {segundos:6.1f} s  {pagina['id']}")


medir("docTR", transcribir_doctr)
medir("Tesseract", transcribir_tesseract)
for nombre, opciones in MODELOS.items():
    modelo, procesador = cargar(**opciones)
    torch.manual_seed(0)
    medir(nombre, lambda imagen: transcribir(modelo, procesador, imagen, **opciones))
    del modelo, procesador  # libera la GPU antes de cargar el modelo siguiente
    gc.collect()
    torch.cuda.empty_cache()

transcripciones = pd.DataFrame(filas)

## 6. CER y WER

- **CER** (*character error rate*): cuántas ediciones de un carácter (sustituir, borrar o agregar) hacen falta para pasar de la transcripción del modelo a la referencia, dividido por la cantidad de caracteres de la referencia. **WER** es lo mismo contado en palabras.
- 0 % es una transcripción perfecta. Puede pasar del 100 % si el modelo agrega texto que no está (por ejemplo, si entra en un bucle y repite una línea).
- Antes de comparar, `limpiar()` quita las marcas de formato, une las palabras cortadas con guion al final de línea y reduce todos los espacios y saltos de línea a un solo espacio. Mayúsculas, tildes y puntuación **sí cuentan**.
- **Mediana**: la página típica. **CER total**: todos los errores sobre todos los caracteres; una sola página mala lo sube mucho.

In [ ]:
import jiwer

referencias = {p["id"]: p["referencia"] for p in paginas}
transcripciones["limpio"] = transcripciones["texto"].map(limpiar)
transcripciones["CER"] = [100 * jiwer.cer(referencias[p], t) for p, t in zip(transcripciones["pagina"], transcripciones["limpio"])]
transcripciones["WER"] = [100 * jiwer.wer(referencias[p], t) for p, t in zip(transcripciones["pagina"], transcripciones["limpio"])]
transcripciones["CER normalizado"] = [100 * jiwer.cer(normalizar(referencias[p]), normalizar(t)) for p, t in zip(transcripciones["pagina"], transcripciones["limpio"])]
transcripciones["WER normalizado"] = [100 * jiwer.wer(normalizar(referencias[p]), normalizar(t)) for p, t in zip(transcripciones["pagina"], transcripciones["limpio"])]

columnas = ["CER", "WER", "CER normalizado", "WER normalizado"]
orden = list(dict.fromkeys(transcripciones["sistema"]))
for columna in columnas:
    print(columna, "por página (%):")
    display(transcripciones.pivot(index="pagina", columns="sistema", values=columna)[orden].round(2))

resumen = transcripciones.groupby("sistema")[columnas + ["segundos"]].median().round(2).add_suffix(" (mediana)")
for sistema, grupo in transcripciones.groupby("sistema"):
    resumen.loc[sistema, "CER total"] = round(100 * jiwer.cer([referencias[p] for p in grupo["pagina"]], list(grupo["limpio"])), 2)
print("Resumen (%; segundos por página):")
display(resumen.sort_values("CER total"))

## 7. Comparar una página

La imagen, la referencia y las transcripciones ya limpias. Cambiá `PAGINA` para ver otra.

In [ ]:
PAGINA = 0  # número de página (0 es la primera)

pagina = paginas[PAGINA]
display(pagina["imagen"])
print("REFERENCIA\n" + pagina["referencia"] + "\n")
for _, fila in transcripciones[transcripciones["pagina"] == pagina["id"]].iterrows():
    print(f"{fila['sistema'].upper()} (CER {fila['CER']:.2f} %)\n{fila['limpio']}\n")

## 8. Qué esperar

| Sistema | CER mediano | CER normalizado mediano | WER normalizado mediano | CER total |
|---|---|---|---|---|
| Churro-3B | 9,6 % | 4,7 % | 15,6 % | 9,4 % |
| LightOnOCR-3-4B | 15,0 % | 12,1 % | 33,8 % | 16,7 % |
| Chandra OCR 2 | 18,6 % | 16,1 % | 40,4 % | 20,3 % |
| Tesseract | 55,5 % | 50,7 % | 94,9 % | 61,4 % |
| docTR | 62,6 % | 59,2 % | 94,7 % | 61,8 % |

Valores medidos al probar este cuaderno en float16, la precisión que se usa en una T4. En Colab pueden variar un poco: otra GPU u otra versión de `transformers`, docTR o Tesseract cambian algún carácter, y LightOnOCR usa muestreo.

**Por qué hay dos CER.** La referencia es diplomática: `ſ`, `q̃`, `v` por `u`, abreviaturas con signos. Un sistema que lee bien pero escribe `s` en lugar de `ſ` suma errores en el CER estricto. El **CER normalizado** compara sin mayúsculas, tildes, puntuación ni esas variantes gráficas; mide mejor si el sistema **leyó** las letras. Las abreviaturas no se desarrollan: si el modelo escribe «que» donde el manuscrito dice «q̃», sigue contando como error.

**Preguntas para discutir:**

- docTR y Tesseract fueron entrenados con letra impresa moderna y no aciertan casi ninguna palabra. ¿Su salida sirve para algo, por ejemplo para encontrar un nombre o una fecha?
- Churro, el único entrenado con manuscritos históricos, comete entre dos veces y media y tres veces y media menos errores que los otros dos modelos. ¿Qué dice eso sobre elegir un modelo por su puntaje en benchmarks generales?
- ¿Los modelos desarrollan abreviaturas, modernizan la ortografía o inventan texto en los tramos difíciles? Compará en la sección 7 la referencia con cada transcripción.
- Churro equivoca una palabra de cada cuatro en el conteo estricto y una de cada siete en el normalizado. ¿Para qué tareas de investigación alcanza: buscar, resumir, citar?

## 9. Guardar los resultados

Se guardan en `resultados/09_ocr_manuscritos/`: el texto crudo de cada sistema en cada página, las métricas por página y el resumen. En Colab se descargan como `.zip`.

In [ ]:
import shutil

SALIDA = Path("resultados") / "09_ocr_manuscritos"
(SALIDA / "textos").mkdir(parents=True, exist_ok=True)
for _, fila in transcripciones.iterrows():
    nombre = re.sub(r"[^\w.-]+", "_", fila["pagina"] + "__" + fila["sistema"]) + ".txt"
    (SALIDA / "textos" / nombre).write_text(fila["texto"], encoding="utf-8")
transcripciones.drop(columns="texto").to_csv(SALIDA / "metricas_por_pagina.csv", index=False)
resumen.to_csv(SALIDA / "resumen.csv")
print("Resultados en", SALIDA)

if EN_COLAB:
    from google.colab import files
    files.download(shutil.make_archive(str(SALIDA), "zip", SALIDA))